<a href="https://colab.research.google.com/github/k-okabe-n/bar_chart_race/blob/main/%E5%8B%95%E3%81%8F%E3%82%B0%E3%83%A9%E3%83%95%E6%94%B9%E8%89%AF%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 必要ライブラリ
# ==========================================

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!apt-get -qq install -y fonts-ipaexfont

!pip install -q japanize-matplotlib

# GitHub最新版をインストール
!pip install -q git+https://github.com/dexplo/bar_chart_race.git

print("インストール完了")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-ipaexfont-gothic.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../fonts-ipaexfont-gothic_00401-3ubuntu1_all.deb ...
Unpacking fonts-ipaexfont-gothic (00401-3ubuntu1) ...
Selecting previously unselected package fonts-ipaexfont-mincho.
Preparing to unpack .../fonts-ipaexfont-mincho_00401-3ubuntu1_all.deb ...
Unpacking fonts-ipaexfont-mincho (00401-3ubuntu1) ...
Selecting previously unselected package fonts-ipaexfont.
Preparing to unpack .../fonts-ipaexfont_00401-3ubuntu1_all.deb ...
Unpacking fonts-ipaexfont (00401-3ubuntu1) ...
Setting up fonts-ipaexfont-mincho (00401-3ubuntu1) ...
update-alternatives: using /usr/share/fonts/opentype/ipaexfont-mincho/ipaexm.ttf to provide /usr/share/fonts/truetype/fonts-japa

In [3]:
# ==========================================
# ライブラリ読み込み
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import japanize_matplotlib
import bar_chart_race as bcr

# ffmpeg
plt.rcParams["animation.ffmpeg_path"] = "/usr/bin/ffmpeg"

# 日本語フォント確認
fonts = sorted(set(f.name for f in fm.fontManager.ttflist))

print("=== 日本語フォント ===")
for f in fonts:
    if "IPA" in f or "Noto" in f:
        print(f)

print()
print("bar_chart_race Version :", bcr.__version__)
print("ライブラリ読込完了")

=== 日本語フォント ===
IPAexGothic

bar_chart_race Version : 0.2.0
ライブラリ読込完了


In [4]:
# ==========================================
# データ読込
# ==========================================

CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/bcr_rawdate.csv"

df = pd.read_csv(
    CSV_PATH,
    encoding="utf_8_sig"
)

df = df.set_index("Date")

# 数値化
for col in df.columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

# 元データ保存
df_original = df.copy()

# 描画用（50億でクリップ）
MAX_VALUE = 5_000_000_000

df_plot = df.clip(upper=MAX_VALUE)

print("読込完了")
print("----------------------------")
print("期間数 :", len(df_plot))
print("部署数 :", len(df_plot.columns))
print()
print("元データ最大")
print(f"{df_original.max().max():,.0f}")
print()
print("描画データ最大")
print(f"{df_plot.max().max():,.0f}")
print()
print("50億超え件数")
print((df_original > MAX_VALUE).sum().sum())

読込完了
----------------------------
期間数 : 361
部署数 : 35

元データ最大
9,575,785,644

描画データ最大
5,000,000,000

50億超え件数
64


In [5]:
# ==========================================
# 描画設定
# ==========================================

# 保存先
OUTPUT_PATH = "/content/drive/MyDrive/my_barchart_race.mp4"

# タイトル
TITLE = "部門別売上推移"

# 表示件数
TOP_N = 20

# グラフサイズ（9:16）
FIGSIZE = (9, 16)

# アニメーション
STEPS = 20
PERIOD_LENGTH = 500

# フォント
FONT = "IPAexGothic"

# 自動色
colors = "dark24"

# matplotlib設定
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False

print("設定完了")

設定完了


In [9]:
import bar_chart_race._bar_chart_race as bcr_internal

# 元の plot_bars を保存（再実行対策）
if not hasattr(bcr_internal._BarChartRace, "_plot_bars_original"):
    bcr_internal._BarChartRace._plot_bars_original = bcr_internal._BarChartRace.plot_bars

def plot_bars_new(self, ax, i):

    # 元の処理
    bcr_internal._BarChartRace._plot_bars_original(self, ax, i)

    # --------------------------
    # 左側（部署名）だけ大きく
    # --------------------------
    for label in ax.get_yticklabels():
        label.set_fontsize(16)

    # --------------------------
    # 下の横軸だけ小さく
    # --------------------------
    for label in ax.get_xticklabels():
        label.set_fontsize(8)

bcr_internal._BarChartRace.plot_bars = plot_bars_new

In [11]:
import bar_chart_race._bar_chart_race as bcr_internal

# 元の関数を保存
if not hasattr(bcr_internal._BarChartRace, "_add_bar_labels_original"):
    bcr_internal._BarChartRace._add_bar_labels_original = (
        bcr_internal._BarChartRace.add_bar_labels
    )

REAL_DF = df_original.copy()

def add_bar_labels_real(self, ax, bar_location, bar_length):

    # まず通常のラベルを描画
    texts = bcr_internal._BarChartRace._add_bar_labels_original(
        self, ax, bar_location, bar_length
    )

    # 現在のフレームの元データ
    row = REAL_DF.loc[self.str_index[min(len(self.str_index)-1, len(REAL_DF)-1)]]

    # 50億超えだけ右側に実数を書く
    for txt, name in zip(texts, self.df_values.columns):

        value = row[name]

        if value > MAX_VALUE:
            txt.set_text(f"▶ {int(value):,}")
            txt.set_fontweight("bold")
            txt.set_color("red")

    return texts

bcr_internal._BarChartRace.add_bar_labels = add_bar_labels_real

In [12]:
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False

print("動画生成を開始します...")
print("（数分かかります）")

bcr.bar_chart_race(

    # データ
    df=df_plot,

    # 保存先
    filename=OUTPUT_PATH,

    # レイアウト
    orientation="h",
    sort="desc",
    n_bars=TOP_N,
    fig_kwargs={
    "figsize": FIGSIZE,
    "dpi": 180
},
    bar_size=0.85,

    # アニメーション
    steps_per_period=STEPS,
    period_length=PERIOD_LENGTH,
    period_label={
    "size":30,
    "x":0.96,
    "y":0.08,
    "ha":"right"
},

    # タイトル
    title={
        "label": TITLE,
        "size": 26
    },

    # 横軸50億固定
    fixed_max=True,

    # 色
    colors=colors,
    filter_column_colors=True,

    # フォント
    shared_fontdict={
        "family": FONT
    },

    tick_label_font={
    "size":8
},

    bar_label_font={
    "size":14
},

    # ラベル
    bar_textposition="outside",
    bar_texttemplate="",

    # 出力
    writer="ffmpeg"
)

print("動画生成完了")

動画生成を開始します...
（数分かかります）


ストリーミング出力は最後の 5000 行に切り捨てられました。
/usr/local/lib/python3.12/dist-packages/bar_chart_race/_bar_chart_race.py:501: UserWarning: Glyph 21942 (\N{CJK UNIFIED IDEOGRAPH-55B6}) missing from font(s) DejaVu Sans.
  ret_val = anim.save(self.filename, fps=self.fps, writer=self.writer,
/usr/local/lib/python3.12/dist-packages/bar_chart_race/_bar_chart_race.py:501: UserWarning: Glyph 26989 (\N{CJK UNIFIED IDEOGRAPH-696D}) missing from font(s) DejaVu Sans.
  ret_val = anim.save(self.filename, fps=self.fps, writer=self.writer,
/usr/local/lib/python3.12/dist-packages/bar_chart_race/_bar_chart_race.py:501: UserWarning: Glyph 25152 (\N{CJK UNIFIED IDEOGRAPH-6240}) missing from font(s) DejaVu Sans.
  ret_val = anim.save(self.filename, fps=self.fps, writer=self.writer,
/usr/local/lib/python3.12/dist-packages/bar_chart_race/_bar_chart_race.py:501: UserWarning: Glyph 26032 (\N{CJK UNIFIED IDEOGRAPH-65B0}) missing from font(s) DejaVu Sans.
  ret_val = anim.save(self.filename, fps=self.fps, writer=self.writer,


動画生成完了
